In [ ]:
# import os
# os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=/home/kiran/cuda_fix'

import tensorflow as tf


2026-03-12 10:01:48.332536: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-12 10:01:48.404013: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-12 10:01:49.922608: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# L = max(0, margin + d(A,P) - d(A,N))  — margin=0.5
def triplet_loss(anchor, positive, negative, margin=0.5):
    d_ap = tf.reduce_sum(tf.square(anchor - positive), axis=-1)
    d_an = tf.reduce_sum(tf.square(anchor - negative), axis=-1)
    return tf.reduce_mean(tf.maximum(0.0, margin + d_ap - d_an))


In [ ]:
# L = -Σ yi log(ŷi)  — binary: 0=authentic, 1=tampered
# Takes logits (pre-sigmoid) — uses stable built-in, no explicit log call
def cross_entropy_loss(y_true, y_pred_logits):
    y_true = tf.cast(tf.reshape(y_true, (-1,)), tf.float32)
    y_pred_logits = tf.reshape(y_pred_logits, (-1,))
    return tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(y_true, y_pred_logits))


In [4]:
# L = BCE + Dice  — pixel-wise mask supervision
def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(tf.reshape(y_true, (-1,)), tf.float32)
    y_pred = tf.cast(tf.reshape(y_pred, (-1,)), tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred)
    dice = (2.0 * intersection + smooth) / (tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth)
    return 1.0 - dice

def segmentation_loss(y_true, y_pred_logits):
    y_true_f = tf.cast(y_true, tf.float32)
    bce = tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(y_true_f, tf.cast(y_pred_logits, tf.float32)))
    y_pred_prob = tf.sigmoid(tf.cast(y_pred_logits, tf.float32))
    return bce + dice_loss(y_true, y_pred_prob)


In [5]:
# L_total = α·L_tri + β·L_ce + γ·L_seg  (default weights all 1.0)
def total_loss(anchor, positive, negative,
               label_true, label_logits,
               mask_true, mask_logits,
               alpha=1.0, beta=1.0, gamma=1.0):
    l_tri = triplet_loss(anchor, positive, negative)
    l_ce  = cross_entropy_loss(label_true, label_logits)
    l_seg = segmentation_loss(mask_true, mask_logits)
    return alpha * l_tri + beta * l_ce + gamma * l_seg
